# ex03 · nb02 — 그리퍼 제어 + 팔·그리퍼 협응(pick&place 흉내) + 플래닝 씬

이 노트북은 mock Piper **팔 + 그리퍼**를 **실시간으로** 움직인다. 셀을 실행하면 RViz 속 로봇이 진짜로 움직여.

## 실행 전 체크
- **노트북 런치 스택이 먼저 떠 있어야 한다.** (`ex03_moveit_py_notebook.launch.py`)
- 컨트롤러가 전부 active 되기까지 **약 30초** 걸린다. RViz 에 로봇이 뜨거나 `ros2 control list_controllers` 가 active 로 나오면 그때 아래 설정 셀을 실행해.

## 딱 하나만 기억해
> **설정 셀(아래 MoveItPy 생성 셀)은 커널당 딱 한 번만 실행한다.**
> MoveItPy 는 자기 프로세스 안에서 move_group 을 직접 띄우기 때문에, 같은 커널에서 두 번 만들면 충돌난다. 셀을 반복 실행하고 싶으면 설정 셀 **말고** 아래 모션 셀들만 다시 돌려.

> 한 커널(= 노트북 한 개)만 이 로봇을 잡을 수 있다. 다른 nb 노트북을 동시에 실행하지 말 것.

이 nb02 는 nb01(팔 기본기)과 달리 **그리퍼**와 **플래닝 씬 충돌물체**까지 같이 다룬다. 그래서 설정 셀에서 `arm` 과 `gripper` **두 개**의 planning component 를 다 가져온다.

In [ ]:
import time

from geometry_msgs.msg import Pose, PoseStamped
from moveit.core.robot_state import RobotState
from moveit.planning import MoveItPy
from moveit_msgs.msg import CollisionObject
from rclpy.logging import get_logger
from shape_msgs.msg import SolidPrimitive
import rclpy

from moveit_py_params import build_moveit_config_dict

# 커널당 딱 한 번. 이미 init 됐으면 건너뛴다.
if not rclpy.ok():
    rclpy.init()

logger = get_logger("nb02_gripper_scene")

# MoveItPy 가 in-process move_group 을 올린다. config_dict 는 런치와 동일한 단일 소스.
robot = MoveItPy(node_name="moveit_py", config_dict=build_moveit_config_dict())

# 이 노트북은 팔과 그리퍼를 둘 다 제어하므로 planning component 를 둘 다 가져온다.
arm = robot.get_planning_component("arm")
gripper = robot.get_planning_component("gripper")
robot_model = robot.get_robot_model()

print("MoveItPy(arm+gripper) 준비 완료")

In [ ]:
def plan_and_execute(robot, component, logger, sleep_time=0.0):
    '''계획→실행 헬퍼 (리포 ex03 과 동일).

    목표(goal)는 이 함수 밖에서 미리 set_goal_state 로 세팅해두고 호출한다.
    '''
    logger.info("계획 중...")
    result = component.plan()
    if result:
        logger.info("실행 중...")
        robot.execute(result.trajectory, controllers=[])  # controllers=[] → 자동 선택
    else:
        logger.error("계획 실패")
    time.sleep(sleep_time)
    return bool(result)

## 1) 그리퍼 — SRDF 이름 상태로 열고/반쯤/닫기

`gripper` 그룹은 관절 하나(`gripper`)짜리다. 범위는 **0.0(완전 닫힘) ~ 0.1(완전 열림)**.

SRDF 에 이름 상태 3개가 정의돼 있어서 값을 외울 필요가 없다:
- `gripper_open` → 0.1 (활짝)
- `gripper_half` → 0.05 (반쯤)
- `gripper_close` → 0.0 (꽉 닫힘)

아래 세 셀을 하나씩 실행하면서 그리퍼가 움직이는 걸 봐.

**그리퍼 열기** (`gripper_open`, 0.1)

In [ ]:
gripper.set_start_state_to_current_state()
gripper.set_goal_state(configuration_name="gripper_open")
plan_and_execute(robot, gripper, logger, sleep_time=1.0)

**그리퍼 반쯤** (`gripper_half`, 0.05)

In [ ]:
gripper.set_start_state_to_current_state()
gripper.set_goal_state(configuration_name="gripper_half")
plan_and_execute(robot, gripper, logger, sleep_time=1.0)

**그리퍼 닫기** (`gripper_close`, 0.0)

In [ ]:
gripper.set_start_state_to_current_state()
gripper.set_goal_state(configuration_name="gripper_close")
plan_and_execute(robot, gripper, logger, sleep_time=1.0)

## 2) 그리퍼 — 임의의 관절값으로 (이름 상태 말고 숫자 직접)

이름 상태는 0.0/0.05/0.1 세 개뿐이지만, 실제로는 **0.0~0.1 사이 아무 값**이나 줄 수 있다. 예를 들어 0.03 만큼만 벌리고 싶다면?

`RobotState` 에 관절값을 넣고 `construct_joint_constraint` 로 목표 제약을 만들어 넘긴다. (팔 관절 목표를 줄 때랑 똑같은 패턴 — 관절이 하나뿐인 그리퍼 그룹에도 그대로 쓴다.)

혹시 이 API 가 이 그룹에서 안 먹으면(단일 관절 그룹 특이케이스) 에러 대신 안내만 찍고 넘어가도록 `try/except` 로 감쌌다. 그럴 땐 위의 이름 상태 방식을 쓰면 된다.

In [ ]:
from moveit.core.kinematic_constraints import construct_joint_constraint

try:
    gripper.set_start_state_to_current_state()

    gs = RobotState(robot_model)
    gs.joint_positions = {"gripper": 0.03}   # 0.0~0.1 사이 임의값

    jc = construct_joint_constraint(
        robot_state=gs,
        joint_model_group=robot_model.get_joint_model_group("gripper"),
    )
    gripper.set_goal_state(motion_plan_constraints=[jc])
    ok = plan_and_execute(robot, gripper, logger, sleep_time=1.0)
    print("raw 관절값 제어 성공" if ok else "계획 실패 — 이름 상태 방식을 써라")
except Exception as exc:
    # 단일 관절 그룹에서 construct_joint_constraint 가 예외를 내면 여기로.
    logger.warn(f"raw 관절값 경로 사용 불가({exc}) — 위의 이름 상태(gripper_open 등)를 써라")

## 3) 팔·그리퍼 협응 — pick & place 흉내내기

이제 팔과 그리퍼를 번갈아 움직여서 "집어서 옮기는" 동작을 흉내낸다. 아래 셀들을 **위에서 아래로 하나씩** 실행하면서 로봇이 한 단계씩 움직이는 걸 지켜봐.

순서:
1. 팔 → `home`
2. 그리퍼 열기
3. 팔 → **집는 자세** (관절 목표)
4. 그리퍼 닫기 (집었다고 치고)
5. 팔 → **옮기는 자세** (다른 관절 목표)
6. 그리퍼 열기 (내려놨다고 치고)
7. 팔 → `home` 복귀

> 팔 관절 목표는 전부 크기 **≤ 1.0 rad** 수준의 온건한 값이라 셀프충돌/한계 걱정 없이 잘 풀린다. `plan_and_execute` 는 목표를 세팅하지 않으니, 팔 목표는 셀 안에서 먼저 `set_goal_state` 로 세팅한 뒤 호출한다.

**[1/7] 팔 → home** (수직 초기자세)

In [ ]:
arm.set_start_state_to_current_state()
arm.set_goal_state(configuration_name="home")
plan_and_execute(robot, arm, logger, sleep_time=1.0)

**[2/7] 그리퍼 열기** — 물체를 잡기 전 미리 벌려둔다

In [ ]:
gripper.set_start_state_to_current_state()
gripper.set_goal_state(configuration_name="gripper_open")
plan_and_execute(robot, gripper, logger, sleep_time=1.0)

**[3/7] 팔 → 집는 자세** — 물체 앞으로 내려가는 관절 목표

In [ ]:
arm.set_start_state_to_current_state()

pick_state = RobotState(robot_model)
pick_state.joint_positions = {
    "joint1": 0.4, "joint2": 0.6, "joint3": -0.6,
    "joint4": 0.0, "joint5": 0.5, "joint6": 0.0,
}
jc = construct_joint_constraint(
    robot_state=pick_state,
    joint_model_group=robot_model.get_joint_model_group("arm"),
)
arm.set_goal_state(motion_plan_constraints=[jc])
plan_and_execute(robot, arm, logger, sleep_time=1.0)

**[4/7] 그리퍼 닫기** — 물체를 집었다고 치고 오므린다

In [ ]:
gripper.set_start_state_to_current_state()
gripper.set_goal_state(configuration_name="gripper_close")
plan_and_execute(robot, gripper, logger, sleep_time=1.0)

**[5/7] 팔 → 옮기는 자세** — 반대편으로 이동하는 다른 관절 목표

In [ ]:
arm.set_start_state_to_current_state()

place_state = RobotState(robot_model)
place_state.joint_positions = {
    "joint1": -0.5, "joint2": 0.5, "joint3": -0.5,
    "joint4": 0.0, "joint5": 0.4, "joint6": 0.0,
}
jc = construct_joint_constraint(
    robot_state=place_state,
    joint_model_group=robot_model.get_joint_model_group("arm"),
)
arm.set_goal_state(motion_plan_constraints=[jc])
plan_and_execute(robot, arm, logger, sleep_time=1.0)

**[6/7] 그리퍼 열기** — 물체를 내려놨다고 치고 벌린다

In [ ]:
gripper.set_start_state_to_current_state()
gripper.set_goal_state(configuration_name="gripper_open")
plan_and_execute(robot, gripper, logger, sleep_time=1.0)

**[7/7] 팔 → home 복귀** — 한 사이클 끝

In [ ]:
arm.set_start_state_to_current_state()
arm.set_goal_state(configuration_name="home")
plan_and_execute(robot, arm, logger, sleep_time=1.0)

## 4) 플래닝 씬 — 충돌물체(collision object)

플래너에게 "여기 장애물 있어" 하고 알려주는 방법. `planning_scene_monitor` 의 `read_write()` 컨텍스트로 씬에 상자를 직접 넣는다.

- `header.frame_id = "base_link"` (베이스 프레임 기준 좌표)
- 상자는 `+y` 쪽(팔 작업영역 옆)에 둬서 `home` 자세를 막지 않는다 → 플래너는 상자를 **인지하되** 계획은 성공한다.

**상자 1개 추가** (`obstacle_box`)

In [ ]:
BOX_ID = "obstacle_box"

psm = robot.get_planning_scene_monitor()
with psm.read_write() as scene:
    obj = CollisionObject()
    obj.header.frame_id = "base_link"
    obj.id = BOX_ID

    box = SolidPrimitive()
    box.type = SolidPrimitive.BOX
    box.dimensions = [0.1, 0.1, 0.2]

    pose = Pose()
    pose.position.x = 0.0
    pose.position.y = 0.30   # 팔 옆쪽
    pose.position.z = 0.20
    pose.orientation.w = 1.0

    obj.primitives.append(box)
    obj.primitive_poses.append(pose)
    obj.operation = CollisionObject.ADD

    scene.apply_collision_object(obj)
    scene.current_state.update()   # 씬 상태 갱신 필수

print(f"충돌물체 '{BOX_ID}' 추가 완료")

**상자가 있는 상태로 home 계획** — 플래너가 상자를 알고도 (안 막히니) 계획 성공

In [ ]:
arm.set_start_state_to_current_state()
arm.set_goal_state(configuration_name="home")
plan_and_execute(robot, arm, logger, sleep_time=1.0)

**두 번째 장애물 추가** — 다른 위치(`-y`)에 상자 하나 더

In [ ]:
BOX2_ID = "obstacle_box_2"

with psm.read_write() as scene:
    obj = CollisionObject()
    obj.header.frame_id = "base_link"
    obj.id = BOX2_ID

    box = SolidPrimitive()
    box.type = SolidPrimitive.BOX
    box.dimensions = [0.1, 0.1, 0.2]

    pose = Pose()
    pose.position.x = 0.0
    pose.position.y = -0.30   # 반대편 옆쪽
    pose.position.z = 0.20
    pose.orientation.w = 1.0

    obj.primitives.append(box)
    obj.primitive_poses.append(pose)
    obj.operation = CollisionObject.ADD

    scene.apply_collision_object(obj)
    scene.current_state.update()

print(f"충돌물체 '{BOX2_ID}' 추가 완료 (이제 씬에 상자 2개)")

**장애물 제거** — 같은 id 에 `REMOVE` 를 주면 씬에서 빠진다 (여기선 상자 2개 다 제거)

In [ ]:
for rid in (BOX_ID, BOX2_ID):
    with psm.read_write() as scene:
        obj = CollisionObject()
        obj.header.frame_id = "base_link"
        obj.id = rid
        obj.operation = CollisionObject.REMOVE   # 같은 id 로 제거
        scene.apply_collision_object(obj)
        scene.current_state.update()
    print(f"충돌물체 '{rid}' 제거 완료")

### RViz 에서 상자가 보일까?

정직하게 말하면 **이 리포의 관전용 RViz(watcher)는 최소 구성**이라 `RobotModel` + `TF` 정도만 켜져 있을 수 있다. 그런 RViz 에서는 방금 넣은 충돌 상자가 **안 보일 수도 있다**.

하지만 안 보인다고 없는 게 아니다 — 상자는 **분명히 플래닝 씬에 들어가 있고, 계획에도 영향을 준다**(상자 위치를 `home` 을 막는 자리로 바꿔서 계획을 실패시켜보면 확인된다).

RViz 에서 눈으로 확인하고 싶으면 RViz 좌하단 **Add → `MotionPlanning`** (또는 `PlanningScene` / `MarkerArray`) 디스플레이를 추가하고, 토픽을 플래닝 씬에 맞추면 상자가 렌더링된다.

### (참고) 물체를 그리퍼에 붙였다 떼기 — attach / detach

진짜 pick&place 에서는 집은 물체를 그리퍼(`tcp_link`)에 **attach** 해서 팔이 움직일 때 물체도 같이 딸려오게(그리고 충돌 계산에 포함되게) 만든다. MoveIt 에는 `AttachedCollisionObject` / 씬 상태 API 로 이걸 하는 방법이 있다.

다만 이 노트북에서는 정확한 호출 시그니처를 **이 환경에서 검증하지 못했기 때문에** 에러 나는 셀을 넣지 않고 안내만 남긴다. 필요하면 `moveit_msgs/AttachedCollisionObject` 와 `scene` 의 attach 관련 API 를 확인해서 추가하면 된다.

## 마무리 — 반복(iteration) 워크플로우

이 노트북의 진짜 이점은 **커널을 살려둔 채 셀만 다시 돌리는 것**이다.

- 그리퍼 관절값(`0.03`)이나 팔 관절 목표(`joint1`, `joint2`...)를 바꾸고 **Shift+Enter** 만 누르면, colcon 재빌드도 런치 재시작도 없이 로봇이 바로 새 동작을 한다.
- 장애물 위치(`pose.position.y` 등)를 `home` 을 막는 자리로 옮겨서 **계획이 실패하는 것**도 실험해봐 — 플래너가 씬을 진짜로 보고 있다는 증거다.

주의:
- **설정 셀(MoveItPy 생성 셀)은 절대 다시 실행하지 마라.** 모션 셀만 반복한다.
- 스택을 내릴 때(런치 Ctrl-C) 커널에서 `Kernel died` / segfault 비슷한 게 떠도 **정상**이다 — MoveItPy 소멸자의 알려진 이슈일 뿐, 작업 결과와 무관하다.
- URDF/SRDF/컨트롤러 YAML 을 바꿨을 때만 런치를 재시작하면 된다.